In [ ]:
# creates_csv_with_actual_name_&_renamed_Name_MULTI_IMAGE_2
import os
import pandas as pd

# ===== PATHS =====
csv_path = r"/home/khushi/Pixonate/New_Anemia/R_new_anemia/overall_data_with_copied_images.csv"
images_folder = r"//home/khushi/Pixonate/New_Anemia/R_new_anemia/overall_main_dataset_found/Conjuctiva"
output_csv = r"/home/khushi/Pixonate/New_Anemia/R_new_anemia/overall_data_with_renamed_images.csv"

hb_col = "HBvalue"
copied_col = "CopiedImagePaths"
# =================

# ---------- HB CLEANER ----------
def clean_hb(val):
    if pd.isna(val):
        return None
    s = str(val).strip()
    s = "".join(c for c in s if c.isdigit() or c == ".")
    if s.count(".") > 1:
        first, *rest = s.split(".")
        s = first + "." + "".join(rest)
    try:
        return round(float(s), 1)
    except:
        return None


# ---------- READ CSV ----------
df = pd.read_csv(csv_path)

# Ensure columns exist
df["actual_image_path"] = ""
df["renamed_image_path"] = ""

counter = 1
renamed = 0
skipped = []

# ---------- RENAME LOOP ----------
for idx, row in df.iterrows():

    if pd.isna(row[copied_col]) or str(row[copied_col]).strip() == "":
        skipped.append("No copied image")
        continue

    hb_val = clean_hb(row[hb_col])
    if hb_val is None:
        skipped.append("Invalid HB")
        continue

    # Split multiple images
    image_paths = [p.strip() for p in str(row[copied_col]).split(",") if p.strip()]

    actual_paths_row = []
    renamed_paths_row = []

    for actual_path in image_paths:

        if not os.path.exists(actual_path):
            skipped.append(actual_path)
            continue

        new_name = f"{counter}.{hb_val}.jpg"
        new_path = os.path.join(images_folder, new_name)

        # Safety: avoid overwrite
        if os.path.exists(new_path):
            skipped.append(new_path)
            counter += 1
            continue

        try:
            os.rename(actual_path, new_path)
            actual_paths_row.append(actual_path)
            renamed_paths_row.append(new_path)
            counter += 1
            renamed += 1
        except Exception as e:
            print(f"❌ Rename failed: {actual_path} -> {e}")

    # Save comma-separated results for that row
    df.at[idx, "actual_image_path"] = ",".join(actual_paths_row)
    df.at[idx, "renamed_image_path"] = ",".join(renamed_paths_row)

# ---------- SAVE ----------
df.to_csv(output_csv, index=False)

print("\n✅ RENAME SUMMARY")
print(f"Total renamed images: {renamed}")
print(f"CSV saved at: {output_csv}")
print(f"Skipped entries: {len(skipped)}")



✅ RENAME SUMMARY
Total renamed images: 1932
CSV saved at: /home/khushi/Pixonate/New_Anemia/R_new_anemia/overall_data_with_renamed_images.csv
Skipped entries: 209
